

# Voice to Notes — Upload a Voice Recording, Get Clean Notes With a Summary

**Project | AI Mini Projects Series**

**Problem:** Meetings, lectures and voice notes are hard to search and nobody wants to listen to them again.

**What this notebook does:** You upload a voice recording. It turns the speech into text with Whisper and then makes structured notes: title, summary, key points, action items, decisions and open questions.

**How it works:** audio → Whisper (speech to text, on Groq) → transcript → LLM → structured notes → save.

**Tools:** Python, Groq Whisper (whisper-large-v3-turbo), Groq LLM (openai/gpt-oss-120b), gTTS (sample audio), Google Colab

## Step 1: Install libraries

In [1]:
!pip install -q openai
# gTTS is installed without its extra packages, because it asks for an old click version that breaks other Colab libraries
!pip install -q --no-deps gTTS

## Step 2: Enter Groq key

In [2]:
import os
import getpass
from openai import OpenAI

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
model_name = "openai/gpt-oss-120b"
print("Groq client ready")

Enter your Groq API key: ··········
Groq client ready


## Step 3: Helper function to call the LLM

In [3]:
def ask_llm(system_msg, user_msg, temperature=0.3):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=temperature
        )
        return response.choices[0].message.content
    except Exception as e:
        return "Error calling Groq: " + str(e)

## Step 4: Upload your audio (or make a sample)
Tip: record a voice note on your phone and upload it here.

In [ ]:
import os

audio_path = None
using_sample = False

try:
    from google.colab import files
    print("Upload your audio file now (cancel to use a sample)")
    uploaded = files.upload()
    if len(uploaded) > 0:
        audio_path = list(uploaded.keys())[0]
except Exception as e:
    print("Upload not available, using a sample")

sample_text = (
    "Hi team, this is the weekly project meeting. First, the website redesign is almost done. "
    "Priya will finish the home page by Friday. Second, we have a budget problem. "
    "The marketing budget was cut by ten percent, so we decided to stop the paid ads for this month. "
    "Rahul will check which free channels we can use instead and share a list by Wednesday. "
    "Third, the customer survey results came in. Most customers like the new app, but many asked for a dark mode. "
    "We decided to add dark mode in the next release. Amit will write the task for the developers today. "
    "One open question is who will handle customer emails while Neha is on leave next week. "
    "Please think about it and reply in the group chat. Thanks everyone, see you next Monday."
)

if audio_path is None:
    using_sample = True
    try:
        from gtts import gTTS
        tts = gTTS(sample_text, lang="en")
        tts.save("sample_meeting.mp3")
        audio_path = "sample_meeting.mp3"
        print("Sample audio created:", audio_path)
    except Exception as e:
        print("Could not make sample audio:", str(e)[:100])

if audio_path is not None:
    size_mb = os.path.getsize(audio_path) / (1024 * 1024)
    print("Audio file:", audio_path, "| size (MB):", round(size_mb, 2))
    if size_mb > 25:
        print("This file is bigger than 25 MB. Please compress it or cut it into smaller parts.")

Upload your audio file now (cancel to use a sample)


## Step 5: Speech to text with Whisper
If this step fails, the sample text is used so the rest of the notebook still runs.

In [5]:
names_text = input("Names or special words in the recording, separated by commas (press Enter to skip): ").strip()

whisper_hint = ""
if names_text != "":
    whisper_hint = "Names and words in this recording: " + names_text + "."

transcript = ""

if audio_path is not None:
    try:
        with open(audio_path, "rb") as f:
            if whisper_hint != "":
                result = client.audio.transcriptions.create(
                    model="whisper-large-v3-turbo",
                    file=f,
                    prompt=whisper_hint
                )
            else:
                result = client.audio.transcriptions.create(
                    model="whisper-large-v3-turbo",
                    file=f
                )
        transcript = result.text.strip()
        print("Transcription done")
    except Exception as e:
        print("Transcription failed:", str(e)[:200])

if transcript == "":
    transcript = sample_text
    print("Using the sample text instead of real transcription")

print("Transcript length (words):", len(transcript.split()))
print("First 400 characters:")
print(transcript[:400])
print()
print("Check the names above. If a name is wrong, run this cell again and type the names.")

Names or special words in the recording, separated by commas (press Enter to skip): 
Transcription done
Transcript length (words): 148
First 400 characters:
Hello everyone, this is the Monday product meeting for the Sunrise app. First, the beta test. 15 users tested the new login screen, and 12 of them said it is easier than the old one. Second, the budget. We have 2 lakh 50,000 rupees for the next quarter. We decided to spend 1 lakh on advertising and keep the rest for server costs. Third, the tasks. Rahul will fix the login bug by Thursday. Mira wil

Check the names above. If a name is wrong, run this cell again and type the names.


## Step 6: Make structured notes

In [6]:
name_rule = ""
if names_text != "":
    name_rule = " (the correct spellings are: " + names_text + ")"

notes_prompt = (
    "Here is the transcript of a voice recording:\n\n" + transcript[:12000] + "\n\n"
    "Make clean notes in markdown with these sections:\n"
    "# (a short title)\n"
    "## Summary (3 lines)\n"
    "## Key Points (bullets)\n"
    "## Action Items (bullets: person, task and deadline. If no deadline was said, write 'no deadline mentioned')\n"
    "## Decisions (bullets)\n"
    "## Open Questions (bullets)\n\n"
    "Strict rules:\n"
    "- Use only what was said. Do not add anything that is not in the transcript.\n"
    "- Do not calculate anything. If the speaker says 'the rest' or 'remaining', write 'the rest' and do not work out a number.\n"
    "- Keep numbers and units exactly as spoken (for example '2 lakh 50,000 rupees').\n"
    "- Decisions are only things the speaker clearly said were decided (for example 'we decided' or 'the date is'). Do not turn feedback or facts into decisions.\n"
    "- Write names exactly as in the transcript" + name_rule + ".\n"
    "- If a section has nothing, write 'None mentioned'."
)

notes = ask_llm("You turn speech transcripts into clear notes.", notes_prompt, 0.2)

from IPython.display import Markdown, display
display(Markdown(notes))

# Sunrise App – Monday Product Meeting  

## Summary  
- Beta test of the new login screen involved 15 users, with 12 finding it easier.  
- Budget for the next quarter is 2 lakh 50,000 rupees; 1 lakh will go to advertising and the rest to server costs.  
- Launch is set for 20 November, and several tasks were assigned with specific deadlines.

## Key Points  
- 15 users tested the new login screen; 12 said it is easier than the old one.  
- Budget: 2 lakh 50,000 rupees for the next quarter.  
- Decision to allocate 1 lakh to advertising and keep the rest for server costs.  
- Rahul will fix the login bug.  
- Mira will contact the 5 beta users who had problems.  
- Sneh will update the pitch deck (date not set).  
- Launch date is the 20th of November.  
- Amit is traveling next week, leaving the help‑page author unknown.

## Action Items  
- **Rahul** – Fix the login bug – **Deadline:** Thursday  
- **Mira** – Contact the 5 beta users who had problems – **Deadline:** today  
- **Sneh** – Update the pitch deck – **Deadline:** no deadline mentioned  

## Decisions  
- Spend 1 lakh on advertising and keep the rest for server costs.  
- Launch date is the 20th of November.  

## Open Questions  
- Who will write the help page, given that Amit is traveling next week?  

## Step 6b: Check the notes against the transcript
A second AI pass lists anything in the notes that the transcript does not clearly support (a calculated number, a decision that was not made, a changed name). It is an AI checking an AI, so it can miss things.

In [7]:
check_prompt = (
    "Transcript:\n" + transcript[:12000] + "\n\nNotes:\n" + notes + "\n\n"
    "Check the notes against the transcript. List every statement in the notes that is NOT clearly supported by the transcript "
    "(for example a number that was calculated, a decision that was not stated, or a name spelled differently). "
    "If everything is supported, reply only: All supported. Otherwise give a short bullet list."
)

check_result = ask_llm("You are a strict checker. You only compare the notes with the transcript.", check_prompt, 0)
print("NOTES CHECK:")
print(check_result)

NOTES CHECK:
All supported.


## Step 7: Save the notes

In [8]:
with open("voice_notes.md", "w", encoding="utf-8") as f:
    f.write(notes + "\n\n---\n## Full Transcript\n\n" + transcript + "\n")
print("Saved voice_notes.md")

try:
    from google.colab import files
    files.download("voice_notes.md")
except Exception as e:
    print("Download not available here, file is saved in the notebook folder")

Saved voice_notes.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 8: Ask a question about the recording
Type a question. Type `exit` to stop.

In [9]:
while True:
    q = input("Ask something about the recording (type exit to stop): ")
    if q.strip().lower() == "exit":
        print("Bye!")
        break
    if q.strip() == "":
        continue
    a = ask_llm(
        "Answer only using the transcript. If it is not there say 'This was not mentioned in the recording'. Keep it short.",
        "Transcript:\n" + transcript[:12000] + "\n\nQuestion: " + q
    )
    print("Answer:", a)
    print("-" * 60)

Ask something about the recording (type exit to stop): What is the meeting about?
Answer: The Monday product meeting for the Sunrise app.
------------------------------------------------------------
Ask something about the recording (type exit to stop): 	How many users tested the login screen?
Answer: 15 users.
------------------------------------------------------------
Ask something about the recording (type exit to stop): What is the budget for the next quarter?
Answer: 2 lakh 50,000 rupees.
------------------------------------------------------------
Ask something about the recording (type exit to stop): How much will be spent on advertising?
Answer: 1 lakh.
------------------------------------------------------------
Ask something about the recording (type exit to stop): 	What will Rahul do, and by when?
Answer: Rahul will fix the login bug by Thursday.
------------------------------------------------------------
Ask something about the recording (type exit to stop): exit
Bye!


## Key Points

- Whisper turns speech into text, and the LLM only does the notes part, so the notes depend on how good the transcript is
- Names are the weak spot. In my first test Whisper heard "Meera" as "Mira" and "Sneha" as "Sneh", and the notes copied the mistakes. Typing the names in Step 5 gives Whisper a hint
- My first notes also added things nobody said: a calculated number (the remaining budget) and a decision that was never made. The notes prompt now forbids calculating and inventing decisions, and Step 6b checks the notes against the transcript. That is an AI checking an AI, so it can miss things. Always check important parts yourself
- Groq has a 25 MB limit for free audio uploads, so long recordings must be compressed or cut
- If transcription fails, the sample text is used, so the notebook does not stop


---
Built by Akshat Kesharwani | GitHub: https://github.com/akshatkesharwani-info | Portfolio: https://akshatkesharwani-info.github.io/